# Swing

Swing Level Breakout (Price Action) \
It is built on level detector code. \
It uses ATR(14) for dynamic stops/tolerance (adapts to volatility). \
Best for scalping (1m-5m) and intraday (15m-1h). Uses dynamic swing highs/lows (like the level detector).

__How Swing Algorithm Determines Entry/Exit:__
- Detects swing highs (resistance) and lows (support) with configurable left/right window.
- Long Entry: Price closes above a significant resistance level + confirmation candle.
- Short Entry: Price closes below a significant support level + confirmation candle.
- Exit Long: Price closes below next support or ATR trailing stop hit.
- Exit Short: Price closes above next resistance or ATR trailing stop hit.
- Ideal for scalping/intraday when price respects liquidity levels.

## Configuration

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [1]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE, save_result

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()

## Swing Breakout

In [ ]:
# Import Swing strategy
from engine.strategies import SwingBreakoutStrategy

In [ ]:
# Backtest Swing strategy
config = StrategyConfig()
strategy = SwingBreakoutStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# Swing strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Inverse Swing

In [4]:
# Import inverse Swing strategy
from engine.strategies import InverseSwingBreakoutStrategy

In [ ]:
# Backtest inverse Swing strategy
config = StrategyConfig()
strategy = InverseSwingBreakoutStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# inverse Swingt strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()